In [4]:
import json
import torch
import random
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm import tqdm

def compute_costs(data_path, cost_model_path="indicator", device="cuda", num_classes=4, n_samples=None):
    """
    Compute costs for a dataset, randomly sampling n prompts if specified
    
    Args:
        data_path: Path to JSONL file containing the dataset
        cost_model_path: Either "indicator" or path to a classification model
        device: Device to run computations on
        num_classes: Number of classes for the cost vector
        n_samples: Number of prompts to randomly sample (None for all prompts)
    
    Returns:
        costs: Tensor of shape (num_samples, num_responses, num_classes) containing computed costs
    """
    # First pass: count total number of examples and collect all data
    all_data = []
    with open(data_path, 'r') as f:
        for line in f:
            if line.strip():
                all_data.append(json.loads(line))
    
    # Random sampling if n_samples is specified
    if n_samples is not None and n_samples < len(all_data):
        selected_data = random.sample(all_data, n_samples)
        print(f"Randomly sampled {n_samples} prompts from {len(all_data)} total")
    else:
        selected_data = all_data
        print(f"Using all {len(all_data)} prompts")

    # Process selected data
    all_prompts = []
    all_responses = []
    for data in selected_data:
        all_prompts.append(data["prompt"])
        all_responses.append(data["generated_texts"])
    
    num_samples = len(all_prompts)
    num_responses = len(all_responses[0])  # Assuming all prompts have same number of responses
    
    # Initialize costs tensor
    costs = torch.zeros(num_samples, num_responses, num_classes)
    costs = costs.to(device)
    
    if cost_model_path == "indicator":
        raise NotImplementedError("Indicator approach requires labels in the dataset")
        
    else:
        print("Loading cost model...")
        cost_model = AutoModelForSequenceClassification.from_pretrained(
            cost_model_path,
            num_labels=num_classes,
            trust_remote_code=True
        )
        tokenizer = AutoTokenizer.from_pretrained(cost_model_path)

        cost_model.eval()
        cost_model.to(device)
        
        for j, prompt in enumerate(tqdm(all_prompts, desc="Processing prompts")):
            # Process each response for this prompt
            for i, response in enumerate(all_responses[j]):
                # Combine prompt and response
                full_text = prompt + response
                
                # Tokenize
                inputs = tokenizer(
                    full_text,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt"
                ).to(device)
                
                # Compute probabilities
                with torch.no_grad():
                    outputs = cost_model(**inputs)
                    probs = torch.softmax(outputs.logits, dim=-1)
                    costs[j, i] = probs[0]  # Store costs for this prompt-response pair
                
        # Free up memory
        del cost_model
        torch.cuda.empty_cache()
    
    return costs

In [ ]:
data_path = "/home/chiche/pd-alignment/cache/beavertails-12k-bal:train_alpaca-7b-reproduced_n12000_nret10_vllm.jsonl"
cost_model_path = "/home/chiche/pd-alignment/output/classifier/google/shieldgemma-2b-5e-4-4"  # or "indicator"

costs = compute_costs(
    data_path=data_path,
    cost_model_path=cost_model_path,
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_classes=4,  # Adjust based on your classification task
    n_samples=None
)

NameError: name 'compute_costs' is not defined

In [32]:
# save costs
torch.save(costs, "cache/sampled_costs_100.pt")